# Lazypredict

In [1]:
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

In [2]:
# Data Preparation
df = pd.read_excel("Final_PM15-Toilet.xlsx")

# Cleaning
df['MDWT'] = pd.to_numeric(df['MDWT'], errors='coerce')
df = df.dropna()

# Cleaning - GSM
df['GSM'] = df['GSM'].astype(str)
df = df[df['GSM'].str.len() <= 4].copy()
df['GSM'] = df['GSM'].str.replace(',', '.')
df['GSM'] = df['GSM'].astype(float)

# Create Pseudo_Mass Feature
df['pseudo_mass'] = (df['Mean_Stock Flow'] * df['Mean_Stock Consistency'])/ df['Mean_Yankee Speed']

# Create Coating-Release Ratio Feature
df['coating_release_ratio'] = df['Mean_Flow Coating'] / df['Mean_Flow Release']

# X Variables
features = [
    '% NBKP',
    'Mean_Load KWH Tickling Refiner',
    'Mean_Creping',
    'Mean_Jet Wire Ratio',
    'GSM',
    'coating_release_ratio'
]
X = df[features]

# Y Variables

y = df['MDT']

In [3]:
X.tail()

,% NBKP,Mean_Load KWH Tickling Refiner,Mean_Creping,Mean_Jet Wire Ratio,GSM,coating_release_ratio
800,0.0,320.397500,23.006510,0.94,16.0,0.632760
801,0.0,314.134920,23.489669,0.94,16.0,0.652042
802,0.0,311.355421,23.135163,0.94,16.0,0.632765
803,0.0,313.115333,22.726395,0.94,16.0,0.632764
804,0.0,314.591462,21.770716,0.94,16.0,0.632761


In [4]:
y.head()

42    752
43    806
44    835
45    950
46    853
Name: MDT, dtype: int64

In [5]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

In [7]:
# Hitung MAPE untuk tiap model
mape_scores = {}

for model_name, y_pred in predictions.items():
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mape_scores[model_name] = mape

# Tambahkan ke tabel hasil
models["MAPE"] = pd.Series(mape_scores)

# Urutkan (semakin kecil semakin baik)
models = models.sort_values(by="MAPE")

In [8]:
print(models)

                               Adjusted R-Squared     R-Squared  \
Model                                                             
ExtraTreesRegressor                  8.794456e-01  8.904051e-01   
GradientBoostingRegressor            8.676836e-01  8.797123e-01   
KNeighborsRegressor                  8.672597e-01  8.793270e-01   
RandomForestRegressor                8.634259e-01  8.758417e-01   
HistGradientBoostingRegressor        8.582872e-01  8.711702e-01   
AdaBoostRegressor                    8.377391e-01  8.524901e-01   
DecisionTreeRegressor                8.342176e-01  8.492887e-01   
ExtraTreeRegressor                   8.307280e-01  8.461164e-01   
PoissonRegressor                     7.997850e-01  8.179864e-01   
HuberRegressor                       7.946912e-01  8.133557e-01   
LassoCV                              7.928267e-01  8.116606e-01   
LassoLarsCV                          7.925472e-01  8.114066e-01   
LassoLars                            7.922063e-01  8.110966e-0

# Regressor

In [10]:
# Import Libraries
import numpy as np

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error)

# Model Dasar
model = ExtraTreesRegressor(random_state=42,n_jobs=-1)

# Grid Search Hyper Parameter
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [8, 10, 15, 20, 30],
    'min_samples_split': [2, 4, 6, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 0.5, 0.7, 1.0],
    'bootstrap': [True, False]
}

# K-Fold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# GGrid Search CV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1,
    verbose=1
)

# Training + Tuning
grid_search.fit(X, y)

# Model Terbaik
best_model = grid_search.best_estimator_
print("Best Parameters:")
print(grid_search.best_params_)

# Prrediksi
y_pred = best_model.predict(X)

# Matrik Evaluasi
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)
epsilon = 1e-8
mape = np.mean(np.abs((y - y_pred) / (y + epsilon))) * 100

# Hasil
print("\n===== HASIL MODEL TERBAIK =====")
print(f"R2    : {r2:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"MAE   : {mae:.4f}")
print(f"MAPE  : {mape:.2f}%")

Fitting 5 folds for each of 1920 candidates, totalling 9600 fits
Best Parameters:
{'bootstrap': False, 'max_depth': 10, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 6, 'n_estimators': 100}

===== HASIL MODEL TERBAIK =====
R2    : 0.9512
RMSE  : 30.2592
MAE   : 23.3019
MAPE  : 3.54%


### Pickle Files

In [ ]:
import joblib

In [ ]:
joblib.dump(model, 'model_pm15-toilet.pkl')

In [ ]:
features_pkl = X.columns.tolist()
joblib.dump(features_pkl, 'features_pm15-toilet.pkl')

### Features Importance

In [ ]:
# Feature Importance
import pandas as pd

feat_imp = pd.DataFrame({
    "Feature": features, 
    "Importance": model.feature_importances_
})

# Urutkan dari terbesar
feat_imp = feat_imp.sort_values(by="Importance", ascending=False)


# Plot
import matplotlib.pyplot as plt
plt.figure()
plt.barh(feat_imp["Feature"], feat_imp["Importance"])
plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance - Random Forest")

plt.tight_layout()
plt.show()

### SHAP Analysis

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=10)

In [ ]:
# Scatter plot dasar - melihat tren arah secara mendetail
shap.plots.scatter(shap_values[:, "Mean_Load KWH Tickling Refiner"])

In [ ]:
shap.plots.scatter(shap_values[:, "pseudo_mass"])

In [ ]:
shap.plots.scatter(shap_values[:, "% NBKP"])

In [ ]:
shap_values = explainer.shap_values(np.array(X_test))
shap.initjs()
i = 0  # index data

shap.force_plot(
    explainer.expected_value,
    shap_values[i],
    X_test.iloc[i]
)